#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from tqdm import tqdm

/n/home07/than157/.conda/envs/llamafactory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load training set of hellaswag
dataset = load_dataset("openlifescienceai/medmcqa", split="validation")
     ###note
     # the "validation" set (n=1413) on huggingface is acutally the test set based on the medmcqa github repo -- same number of samples
     # the "test" set on huggingface does not provide correct answers, the "cop" column is all -1

#print dataset info
print("Dataset info:")
print(dataset)

#convert to dataframe
df = dataset.to_pandas()

print("# samples:", df.shape[0])

df.head()


Dataset info:
Dataset({
    features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
    num_rows: 4183
})
# samples: 4183


,id,question,opa,opb,opc,opd,cop,choice_type,exp,subject_name,topic_name
0,45258d3d-b974-44dd-a161-c3fccbdadd88,Which of the following is not true for myelina...,Impulse through myelinated fibers is slower th...,Membrane currents are generated at nodes of Ra...,Saltatory conduction of impulses is seen,Local anesthesia is effective only when the ne...,0,multi,None,Physiology,None
1,b944ada9-d776-4c2a-9180-3ae5f393f72d,Which of the following is not true about glome...,The oncotic pressure of the fluid leaving the ...,Glucose concentration in the capillaries is th...,Constriction of afferent aeriole decreases the...,Hematocrit of the fluid leaving the capillarie...,0,multi,Ans-a. The oncotic pressure of the fluid leavi...,Physiology,None
2,b64a9cd7-d076-4c55-8be1-f9c44fece6cc,A 29 yrs old woman with a pregnancy of 17 week...,No test is required now as her age is below 35...,Ultra sound at this point of time will definit...,Amniotic fluid samples plus chromosomal analys...,blood screening at this point of time will cle...,2,single,None,Medicine,None
3,c6365cce-507c-40f6-90a2-46b867f47b6e,Axonal transport is:,Antegrade,Retrograde,Antegrade and retrograde,None,2,multi,Fast anterograde (400 mm/day) transport occurs...,Physiology,None
4,72c1c5e0-b64f-4eef-bf22-ecfb60c5c19c,Low insulin to glucagon ratio is seen in all o...,Glycogen synthesis,Glycogen breakdown,Gluconeogenesis,Ketogenesis,0,multi,Answer- A. Glycogen synthesisLow insulin to gl...,Biochemistry,None


In [3]:
### format question with options

def write_full_question(row):
    full_question = f"{row['question']}\nOptions:\nA. {row['opa']}\nB. {row['opb']}\nC. {row['opc']}\nD. {row['opd']}"
    return full_question

df['full_question'] = df.apply(write_full_question, axis=1)

In [4]:
#look at example, sanity check
print(df.iloc[2]['full_question'])

A 29 yrs old woman with a pregnancy of 17 week has a 10 years old boy with down syndrome. She does not want another down syndrome kid; best advice to her is
Options:
A. No test is required now as her age is below 35 years
B. Ultra sound at this point of time will definitely tell her that next baby will be down syndromic or not
C. Amniotic fluid samples plus chromosomal analysis will definitely tell her that next baby will be down syndromic or not
D. blood screening at this point of time will clear the exact picture


In [5]:
#create column with correct answer
answer_dict = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df['correct_answer'] = df['cop'].map(answer_dict)

#create a column with question id
df['question_id'] = 'medmcqa-' + df.index.astype(str) + '-' + df['subject_name']
df.head()

,id,question,opa,opb,opc,opd,cop,choice_type,exp,subject_name,topic_name,full_question,correct_answer,question_id
0,45258d3d-b974-44dd-a161-c3fccbdadd88,Which of the following is not true for myelina...,Impulse through myelinated fibers is slower th...,Membrane currents are generated at nodes of Ra...,Saltatory conduction of impulses is seen,Local anesthesia is effective only when the ne...,0,multi,None,Physiology,None,Which of the following is not true for myelina...,A,medmcqa-0-Physiology
1,b944ada9-d776-4c2a-9180-3ae5f393f72d,Which of the following is not true about glome...,The oncotic pressure of the fluid leaving the ...,Glucose concentration in the capillaries is th...,Constriction of afferent aeriole decreases the...,Hematocrit of the fluid leaving the capillarie...,0,multi,Ans-a. The oncotic pressure of the fluid leavi...,Physiology,None,Which of the following is not true about glome...,A,medmcqa-1-Physiology
2,b64a9cd7-d076-4c55-8be1-f9c44fece6cc,A 29 yrs old woman with a pregnancy of 17 week...,No test is required now as her age is below 35...,Ultra sound at this point of time will definit...,Amniotic fluid samples plus chromosomal analys...,blood screening at this point of time will cle...,2,single,None,Medicine,None,A 29 yrs old woman with a pregnancy of 17 week...,C,medmcqa-2-Medicine
3,c6365cce-507c-40f6-90a2-46b867f47b6e,Axonal transport is:,Antegrade,Retrograde,Antegrade and retrograde,None,2,multi,Fast anterograde (400 mm/day) transport occurs...,Physiology,None,Axonal transport is:\nOptions:\nA. Antegrade\n...,C,medmcqa-3-Physiology
4,72c1c5e0-b64f-4eef-bf22-ecfb60c5c19c,Low insulin to glucagon ratio is seen in all o...,Glycogen synthesis,Glycogen breakdown,Gluconeogenesis,Ketogenesis,0,multi,Answer- A. Glycogen synthesisLow insulin to gl...,Biochemistry,None,Low insulin to glucagon ratio is seen in all o...,A,medmcqa-4-Biochemistry


## save relevant columns in df to proper format

In [6]:
### save a small sample of dataset

### create jsonl file

#format data for sft
data = []

# for idx in tqdm(range(df.shape[0])):

# Write JSONL file
with open("data/cooked/medmcqa.jsonl", "w", encoding="utf-8") as f:
    for idx in tqdm(range(df.shape[0])):
        row = df.iloc[idx]
        item = {
            "id": str(row["question_id"]),
            "problem": str(row["full_question"]),
            "gt_solution": str(row["correct_answer"]), #f"Answer: {row['correct_answer']}",
            "gt_answer": str(row["correct_answer"])
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Final # of samples in json file:", len(data))
print("Complete!")

  0%|          | 0/4183 [00:00<?, ?it/s]

100%|██████████| 4183/4183 [00:00<00:00, 25765.18it/s]

Final # of samples in json file: 0
Complete!


In [6]:
pwd

'/n/holylabs/doshi-velez_lab/Users/than157/large_projects/learn-better/evolm/evaluation/cot-eval-harness'